In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
load_dotenv()

True

In [8]:
class ParentState(TypedDict):
    question:str
    answer_in_eng: str
    answer_in_hindi: str

In [9]:
parent_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
subbraph_llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

In [10]:
def translation(state: ParentState):
    prompt = f"""
    translate the following text to hindi. keep at natural and clear. dont add extra content.

    text:{state["answer_in_eng"]}
    """.strip()

    translated_text = parent_llm.invoke(prompt).content

    return {'answer_in_hindi':translated_text}

In [14]:
subgraph = StateGraph(ParentState)

subgraph.add_node('translation',translation)

subgraph.add_edge(START, 'translation')
subgraph.add_edge('translation', END)

In [16]:
childgraph = subgraph.compile()

In [17]:
def generate_answer(state: ParentState):
    answer = parent_llm.invoke(f"you are helpful assistant.Answer Clearly. \n\nQuestion: {state['question']}").content
    return {'answer_in_eng':answer}

In [22]:
main_graph = StateGraph(ParentState)

main_graph.add_node('generate_answer',generate_answer)
main_graph.add_node('translate',childgraph)

main_graph.add_edge(START, 'generate_answer')
main_graph.add_edge('generate_answer','translate')
main_graph.add_edge('generate_answer',END)

In [24]:
graph = main_graph.compile()

In [25]:
graph.invoke({'question':'What is quantum physics'})

{'question': 'What is quantum physics',
 'answer_in_eng': 'Quantum physics, also known as quantum mechanics, is a fundamental theory in physics that describes the behavior of matter and energy at the atomic and subatomic levels.\n\nHere\'s a breakdown of what that means clearly:\n\n1.  **The Physics of the Very Small:**\n    *   Classical physics (like Newton\'s laws or Maxwell\'s equations) works incredibly well for objects we encounter in everyday life – planets, cars, baseballs.\n    *   However, when you shrink down to the size of atoms, electrons, photons, and other tiny particles, classical physics breaks down. Quantum physics is the framework that accurately explains how these tiny particles behave.\n\n2.  **Key Concepts (Why it\'s "Quantum"):**\n    *   **Quantization:** This is where the name comes from. In the quantum world, many physical properties (like energy, momentum, or angular momentum) are not continuous but come in discrete, indivisible packets called "quanta." Think